<a href="https://colab.research.google.com/github/manavtyagi931935-boop/operatingsystem/blob/main/manavtyagi_os_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Operating Systems Lab Assignment - Detailed Solutions
This notebook contains the complete, direct executable scripts for all 11 programs, aligned with the exact parameters and logics described in the OS lab manual.

### Program 1: Windows/Linux System Calls
**Aim:** To write a program to simulate process creation using system calls (`fork`, `exec`, `wait`).

Since direct `os.fork()` is restricted by multithreaded notebook environments and can hang the Jupyter container, this implementation simulates the fork branch states directly using the standard `multiprocessing` library to execute code in a separate process space, ensuring a robust, executable process hierarchy.

In [1]:
import os
import sys
import time
from multiprocessing import Process

def child_task():
    print(f"[Child Process] Hello, I am the child. My PID is {os.getpid()} and my Parent's PID is {os.getppid()}")
    print("[Child Process] Executing system directory listing command...")
    sys.stdout.flush()
    os.system("ls -l")
    print("[Child Process] Execution finished. Exiting.")

if __name__ == '__main__':
    print(f"[Parent Process] Hello, I am the parent. My PID is {os.getpid()}")
    print("[Parent Process] Forking/creating a child process...")

    p = Process(target=child_task)
    p.start()

    print("[Parent Process] Waiting for the child process to complete...")
    p.join()

    print(f"[Parent Process] Child process joined with exit status code: {p.exitcode}")
    print("[Parent Process] Exiting successfully.")

[Parent Process] Hello, I am the parent. My PID is 1638
[Parent Process] Forking/creating a child process...
[Child Process] Hello, I am the child. My PID is 3400 and my Parent's PID is 1638
[Child Process] Executing system directory listing command...
[Child Process] Execution finished. Exiting.
[Parent Process] Waiting for the child process to complete...
[Parent Process] Child process joined with exit status code: 0
[Parent Process] Exiting successfully.


### Program 2: Multi-threading using POSIX Threads
**Aim:** To write a program to simulate thread creation and synchronization using POSIX threads or equivalent Python standard `threading` API.

In [2]:
import threading
import time

shared_counter = 0
counter_lock = threading.Lock()

def increment_counter(thread_name):
    global shared_counter
    for i in range(5):
        time.sleep(0.02)
        with counter_lock:
            local_val = shared_counter
            local_val += 1
            shared_counter = local_val
            print(f"[{thread_name}] Incremented counter to {shared_counter}")

t1 = threading.Thread(target=increment_counter, args=("Thread-1",))
t2 = threading.Thread(target=increment_counter, args=("Thread-2",))

print("Starting threads...")
t1.start()
t2.start()

t1.join()
t2.join()

print(f"All threads finished. Final value of the shared counter: {shared_counter}")

Starting threads...
[Thread-1] Incremented counter to 1
[Thread-2] Incremented counter to 2
[Thread-1] Incremented counter to 3
[Thread-2] Incremented counter to 4
[Thread-1] Incremented counter to 5
[Thread-2] Incremented counter to 6
[Thread-1] Incremented counter to 7
[Thread-2] Incremented counter to 8
[Thread-1] Incremented counter to 9
[Thread-2] Incremented counter to 10
All threads finished. Final value of the shared counter: 10


### Program 3: CPU Scheduling Algorithms (FCFS)
**Aim:** To write a program to simulate CPU scheduling algorithms like First Come First Served (FCFS).

In [17]:
class Process:
    def __init__(self, pid, at, bt, pr):
        self.pid = pid
        self.at = at
        self.bt = bt
        self.pr = pr
        self.ct = 0
        self.tat = 0
        self.wt = 0
        self.rem = bt

def print_result(title, p):
    n = len(p)
    total_wt = total_tat = 0
    print(f"\n--- {title} Scheduling Result ---")
    print(f"{'PID':<5}{'AT':<4}{'BT':<4}{'CT':<4}{'TAT':<5}{'WT'}")
    for proc in sorted(p, key=lambda x: x.pid):
        print(f"{proc.pid:<5}{proc.at:<4}{proc.bt:<4}{proc.ct:<4}{proc.tat:<5}{proc.wt}")
        total_wt += proc.wt
        total_tat += proc.tat
    print(f"Average Waiting Time    = {total_wt / n:.2f}")
    print(f"Average Turnaround Time = {total_tat / n:.2f}")

# Helper to reset processes for each run
def get_sample_processes():
    # Processes as defined in the Lab Manual output sample:
    # Process 1: 0 5 2 (PID, AT, BT, Priority)
    # Process 2: 1 3 1
    # Process 3: 2 8 3
    # Process 4: 3 6 4
    return [
        Process(1, 0, 5, 2),
        Process(2, 1, 3, 1),
        Process(3, 2, 8, 3),
        Process(4, 3, 6, 4)
    ]

# 1. FCFS Scheduling
p_fcfs = get_sample_processes()
p_fcfs.sort(key=lambda x: x.at)
time = 0
for proc in p_fcfs:
    if time < proc.at:
        time = proc.at
    time += proc.bt
    proc.ct = time
    proc.tat = proc.ct - proc.at
    proc.wt = proc.tat - proc.bt
print_result("FCFS", p_fcfs)

# 2. SJF (Non-preemptive) Scheduling
p_sjf = get_sample_processes()
n = len(p_sjf)
completed, time = 0, 0
done = [False] * n
while completed < n:
    idx, min_bt = -1, 10**9
    for i in range(n):
        if not done[i] and p_sjf[i].at <= time and p_sjf[i].bt < min_bt:
            min_bt = p_sjf[i].bt
            idx = i
    if idx == -1:
        time += 1
        continue
    time += p_sjf[idx].bt
    p_sjf[idx].ct = time
    p_sjf[idx].tat = p_sjf[idx].ct - p_sjf[idx].at
    p_sjf[idx].wt = p_sjf[idx].tat - p_sjf[idx].bt
    done[idx] = True
    completed += 1
print_result("SJF (Non-preemptive)", p_sjf)

# 3. Priority (Non-preemptive) Scheduling
p_priority = get_sample_processes()
n = len(p_priority)
completed, time = 0, 0
done = [False] * n
while completed < n:
    idx, best = -1, 10**9
    for i in range(n):
        if not done[i] and p_priority[i].at <= time and p_priority[i].pr < best:
            best = p_priority[i].pr
            idx = i
    if idx == -1:
        time += 1
        continue
    time += p_priority[idx].bt
    p_priority[idx].ct = time
    p_priority[idx].tat = p_priority[idx].ct - p_priority[idx].at
    p_priority[idx].wt = p_priority[idx].tat - p_priority[idx].bt
    done[idx] = True
    completed += 1
print_result("Priority (Non-preemptive)", p_priority)

# 4. Round Robin Scheduling
p_rr = get_sample_processes()
quantum = 4
n = len(p_rr)
time, completed = 0, 0
for proc in p_rr:
    proc.rem = proc.bt

queue = []
in_queue = [False] * n

# start with processes arriving at time 0
for i in range(n):
    if p_rr[i].at <= time:
        queue.append(i)
        in_queue[i] = True

while completed < n:
    if not queue:
        time += 1
        for i in range(n):
            if not in_queue[i] and p_rr[i].rem > 0 and p_rr[i].at <= time:
                queue.append(i)
                in_queue[i] = True
        continue
    idx = queue.pop(0)
    slice_ = min(p_rr[idx].rem, quantum)
    time += slice_
    p_rr[idx].rem -= slice_

    for i in range(n):
        if not in_queue[i] and p_rr[i].rem > 0 and p_rr[i].at <= time and i != idx:
            queue.append(i)
            in_queue[i] = True

    if p_rr[idx].rem > 0:
        queue.append(idx)
    else:
        p_rr[idx].ct = time
        p_rr[idx].tat = p_rr[idx].ct - p_rr[idx].at
        p_rr[idx].wt = p_rr[idx].tat - p_rr[idx].bt
        completed += 1
        in_queue[idx] = False

print_result(f"Round Robin (Quantum={quantum})", p_rr)



--- FCFS Scheduling Result ---
PID  AT  BT  CT  TAT  WT
1    0   5   5   5    0
2    1   3   8   7    4
3    2   8   16  14   6
4    3   6   22  19   13
Average Waiting Time    = 5.75
Average Turnaround Time = 11.25

--- SJF (Non-preemptive) Scheduling Result ---
PID  AT  BT  CT  TAT  WT
1    0   5   5   5    0
2    1   3   8   7    4
3    2   8   22  20   12
4    3   6   14  11   5
Average Waiting Time    = 5.25
Average Turnaround Time = 10.75

--- Priority (Non-preemptive) Scheduling Result ---
PID  AT  BT  CT  TAT  WT
1    0   5   5   5    0
2    1   3   8   7    4
3    2   8   16  14   6
4    3   6   22  19   13
Average Waiting Time    = 5.75
Average Turnaround Time = 11.25

--- Round Robin (Quantum=4) Scheduling Result ---
PID  AT  BT  CT  TAT  WT
1    0   5   16  16   11
2    1   3   7   6    3
3    2   8   20  18   10
4    3   6   22  19   13
Average Waiting Time    = 9.25
Average Turnaround Time = 14.75


### Program 4: Process Synchronization (Producer-Consumer)
**Aim:** To write a program to simulate Process Synchronization using Semaphores (Producer-Consumer problem).

In [4]:
import threading
import time
import random

BUFFER_SIZE = 5
buffer = []

empty_sem = threading.Semaphore(BUFFER_SIZE)
full_sem = threading.Semaphore(0)
mutex_lock = threading.Lock()

def producer():
    for i in range(5):
        item = random.randint(1, 100)
        empty_sem.acquire()
        with mutex_lock:
            buffer.append(item)
            print(f"[Producer] Produced item: {item}. Buffer state: {buffer}")
        full_sem.release()
        time.sleep(random.uniform(0.05, 0.15))

def consumer():
    for i in range(5):
        full_sem.acquire()
        with mutex_lock:
            item = buffer.pop(0)
            print(f"[Consumer] Consumed item: {item}. Buffer state: {buffer}")
        empty_sem.release()
        time.sleep(random.uniform(0.08, 0.2))

prod_thread = threading.Thread(target=producer)
cons_thread = threading.Thread(target=consumer)

prod_thread.start()
cons_thread.start()

prod_thread.join()
cons_thread.join()
print("Producer-Consumer simulation completed successfully.")

[Producer] Produced item: 34. Buffer state: [34]
[Consumer] Consumed item: 34. Buffer state: []
[Producer] Produced item: 91. Buffer state: [91]
[Consumer] Consumed item: 91. Buffer state: []
[Producer] Produced item: 77. Buffer state: [77]
[Producer] Produced item: 81. Buffer state: [77, 81]
[Producer] Produced item: 70. Buffer state: [77, 81, 70]
[Consumer] Consumed item: 77. Buffer state: [81, 70]
[Consumer] Consumed item: 81. Buffer state: [70]
[Consumer] Consumed item: 70. Buffer state: []
Producer-Consumer simulation completed successfully.


### Program 5: Readers-Writers Problem
**Aim:** To write a program to simulate the Readers-Writers Problem using mutex locks and condition variables.

In [5]:
import threading
import time

shared_database = 100
reader_count = 0

count_mutex = threading.Lock()
write_block_lock = threading.Lock()

def reader_task(reader_id):
    global reader_count
    time.sleep(0.02 * reader_id)

    with count_mutex:
        reader_count += 1
        if reader_count == 1:
            write_block_lock.acquire()

    print(f"[Reader {reader_id}] Reading shared database value: {shared_database}")
    time.sleep(0.05)

    with count_mutex:
        reader_count -= 1
        if reader_count == 0:
            write_block_lock.release()
    print(f"[Reader {reader_id}] Finished reading.")

def writer_task(writer_id):
    global shared_database
    time.sleep(0.03 * writer_id)

    print(f"[Writer {writer_id}] Requesting write permission...")
    with write_block_lock:
        old_val = shared_database
        shared_database += 50
        print(f"[Writer {writer_id}] Updated database value from {old_val} to {shared_database}")
        time.sleep(0.05)
    print(f"[Writer {writer_id}] Finished writing and released lock.")

threads = []
for i in range(3):
    threads.append(threading.Thread(target=reader_task, args=(i+1,)))
for i in range(2):
    threads.append(threading.Thread(target=writer_task, args=(i+1,)))

for t in threads:
    t.start()
for t in threads:
    t.join()
print("Readers-Writers synchronization simulation complete.")

[Reader 1] Reading shared database value: 100
[Writer 1] Requesting write permission...
[Reader 2] Reading shared database value: 100
[Reader 3] Reading shared database value: 100
[Writer 2] Requesting write permission...
[Reader 1] Finished reading.
[Reader 2] Finished reading.
[Reader 3] Finished reading.
[Writer 1] Updated database value from 100 to 150
[Writer 1] Finished writing and released lock.
[Writer 2] Updated database value from 150 to 200
[Writer 2] Finished writing and released lock.
Readers-Writers synchronization simulation complete.


### Program 6: Dining Philosophers Problem
**Aim:** To write a program to simulate the Dining Philosophers Problem using thread locking configurations.

In [6]:
import threading
import time

num_philosophers = 5
forks = [threading.Lock() for _ in range(num_philosophers)]

def philosopher_lifecycle(philosopher_id):
    for cycle in range(1):
        print(f"[Philosopher {philosopher_id}] is thinking...")
        time.sleep(0.02)

        left_fork = philosopher_id
        right_fork = (philosopher_id + 1) % num_philosophers

        first_fork, second_fork = (left_fork, right_fork) if left_fork < right_fork else (right_fork, left_fork)

        print(f"[Philosopher {philosopher_id}] is hungry. Waiting for forks...")
        with forks[first_fork]:
            with forks[second_fork]:
                print(f"[Philosopher {philosopher_id}] picked up forks and is eating...")
                time.sleep(0.05)
        print(f"[Philosopher {philosopher_id}] finished eating and put down both forks.")

threads = [threading.Thread(target=philosopher_lifecycle, args=(i,)) for i in range(num_philosophers)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print("Dining Philosophers problem finished processing cleanly.")

[Philosopher 0] is thinking...
[Philosopher 1] is thinking...
[Philosopher 2] is thinking...
[Philosopher 3] is thinking...
[Philosopher 4] is thinking...
[Philosopher 0] is hungry. Waiting for forks...
[Philosopher 0] picked up forks and is eating...
[Philosopher 1] is hungry. Waiting for forks...
[Philosopher 2] is hungry. Waiting for forks...
[Philosopher 2] picked up forks and is eating...
[Philosopher 3] is hungry. Waiting for forks...
[Philosopher 4] is hungry. Waiting for forks...
[Philosopher 0] finished eating and put down both forks.
[Philosopher 4] picked up forks and is eating...
[Philosopher 2] finished eating and put down both forks.
[Philosopher 1] picked up forks and is eating...
[Philosopher 4] finished eating and put down both forks.
[Philosopher 3] picked up forks and is eating...
[Philosopher 1] finished eating and put down both forks.
[Philosopher 3] finished eating and put down both forks.
Dining Philosophers problem finished processing cleanly.


### Program 7: Deadlock Avoidance (Banker's Algorithm)
**Aim:** To write a program to simulate deadlock avoidance using the Banker's Algorithm.

In [7]:
num_processes = 5
num_resources = 3

allocation = [
    [0, 1, 0],
    [2, 0, 0],
    [3, 0, 2],
    [2, 1, 1],
    [0, 0, 2]
]

max_claim = [
    [7, 5, 3],
    [3, 2, 2],
    [9, 0, 2],
    [2, 2, 2],
    [4, 3, 3]
]

available = [3, 3, 2]

need = [[max_claim[i][j] - allocation[i][j] for j in range(num_resources)] for i in range(num_processes)]
finish = [False] * num_processes
safe_sequence = []
work = list(available)

print("Initial Need Matrix:")
for i in range(num_processes):
    print(f"P{i}: {need[i]}")

count = 0
while count < num_processes:
    found = False
    for p in range(num_processes):
        if not finish[p]:
            if all(need[p][r] <= work[r] for r in range(num_resources)):
                for r in range(num_resources):
                    work[r] += allocation[p][r]
                finish[p] = True
                safe_sequence.append(p)
                found = True
                count += 1
                print(f"Process P{p} runs to completion. New Available Work: {work}")
                break
    if not found:
        print("System is not in a safe state! Deadlock might occur.")
        break
else:
    print(f"\nSystem is in a SAFE STATE! Safe Sequence: {safe_sequence}")

Initial Need Matrix:
P0: [7, 4, 3]
P1: [1, 2, 2]
P2: [6, 0, 0]
P3: [0, 1, 1]
P4: [4, 3, 1]
Process P1 runs to completion. New Available Work: [5, 3, 2]
Process P3 runs to completion. New Available Work: [7, 4, 3]
Process P0 runs to completion. New Available Work: [7, 5, 3]
Process P2 runs to completion. New Available Work: [10, 5, 5]
Process P4 runs to completion. New Available Work: [10, 5, 7]

System is in a SAFE STATE! Safe Sequence: [1, 3, 0, 2, 4]


### Program 8: Contiguous Memory Allocation (First Fit)
**Aim:** To write a program to simulate contiguous memory allocation using the First Fit technique.

In [12]:
block_sizes = [100, 500, 200, 300, 600]
process_sizes = [212, 417, 112, 426]

num_blocks = len(block_sizes)
num_processes = len(process_sizes)

allocation = [-1] * num_processes
remaining_blocks = list(block_sizes)

for i in range(num_processes):
    for j in range(num_blocks):
        if remaining_blocks[j] >= process_sizes[i]:
            allocation[i] = j
            remaining_blocks[j] -= process_sizes[i]
            break

print("Process No.\tProcess Size\tBlock Allocated\tRemaining Block Size")
for i in range(num_processes):
    alloc_block = allocation[i] + 1 if allocation[i] != -1 else "Not Allocated"
    print(f"{i+1}\t\t{process_sizes[i]}\t\t{alloc_block}\t\t{remaining_blocks[allocation[i]] if allocation[i] != -1 else 'N/A'}")

Process No.	Process Size	Block Allocated	Remaining Block Size
1		212		2		176
2		417		5		183
3		112		2		176
4		426		Not Allocated		N/A


### Program 9: FIFO Page Replacement Algorithm
**Aim:** To write a program to simulate FIFO Page Replacement algorithm.

In [9]:
reference_string = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2, 1, 2, 0, 1, 7, 0, 1]
num_frames = 3

frames = []
page_faults = 0
page_hits = 0

print("Incoming\tFrame State\tResult")
for page in reference_string:
    if page in frames:
        page_hits += 1
        result = "HIT"
    else:
        page_faults += 1
        result = "FAULT"
        if len(frames) < num_frames:
            frames.append(page)
        else:
            frames.pop(0)
            frames.append(page)
    print(f"{page}\t\t{frames}\t\t{result}")

print(f"\nTotal Page Faults: {page_faults}")
print(f"Total Page Hits: {page_hits}")

Incoming	Frame State	Result
7		[7]		FAULT
0		[7, 0]		FAULT
1		[7, 0, 1]		FAULT
2		[0, 1, 2]		FAULT
0		[0, 1, 2]		HIT
3		[1, 2, 3]		FAULT
0		[2, 3, 0]		FAULT
4		[3, 0, 4]		FAULT
2		[0, 4, 2]		FAULT
3		[4, 2, 3]		FAULT
0		[2, 3, 0]		FAULT
3		[2, 3, 0]		HIT
2		[2, 3, 0]		HIT
1		[3, 0, 1]		FAULT
2		[0, 1, 2]		FAULT
0		[0, 1, 2]		HIT
1		[0, 1, 2]		HIT
7		[1, 2, 7]		FAULT
0		[2, 7, 0]		FAULT
1		[7, 0, 1]		FAULT

Total Page Faults: 15
Total Page Hits: 5


### Program 10: SSTF Disk Scheduling Algorithm
**Aim:** To write a program to simulate the Shortest Seek Time First (SSTF) disk scheduling algorithm.

In [10]:
request_queue = [98, 183, 37, 122, 14, 124, 65, 67]
initial_head = 53

pending_requests = list(request_queue)
current_head = initial_head
total_seek_time = 0
seek_sequence = []

print(f"Starting head at: {initial_head}")
while len(pending_requests) > 0:
    closest_request = pending_requests[0]
    min_distance = abs(closest_request - current_head)

    for req in pending_requests:
        dist = abs(req - current_head)
        if dist < min_distance:
            min_distance = dist
            closest_request = req

    total_seek_time += min_distance
    current_head = closest_request
    seek_sequence.append(current_head)
    pending_requests.remove(closest_request)

print(f"Execution Sequence of Requests: {seek_sequence}")
print(f"Total Seek Operations (Distance): {total_seek_time}")

Starting head at: 53
Execution Sequence of Requests: [65, 67, 37, 14, 98, 122, 124, 183]
Total Seek Operations (Distance): 236


### Program 11: File Allocation Methods (Indexed Allocation)
**Aim:** To write a program to simulate Indexed File Allocation.

In [11]:
import random

TOTAL_DISK_BLOCKS = 50
disk_blocks = [0] * TOTAL_DISK_BLOCKS

# Mocking files to be saved
files = {
    "assignment.pdf": 4,  # needs 4 blocks
    "notes.txt": 2        # needs 2 blocks
}

print(f"--- Initializing Disk with {TOTAL_DISK_BLOCKS} blocks ---")
for filename, length in files.items():
    # Choose index block randomly from empty blocks
    empty_indices = [i for i, b in enumerate(disk_blocks) if b == 0]
    index_block = random.choice(empty_indices)
    disk_blocks[index_block] = 1 # Mark index block as occupied

    # Choose data blocks
    empty_indices = [i for i, b in enumerate(disk_blocks) if b == 0]
    if len(empty_indices) < length:
        print(f"Error: Not enough disk blocks to allocate '{filename}'!")
        continue

    data_blocks = random.sample(empty_indices, length)
    for db in data_blocks:
        disk_blocks[db] = 1

    print(f"\nAllocated File: '{filename}'")
    print(f"Index Block: {index_block}")
    print(f"Indexed Blocks pointing to Data Blocks: {data_blocks}")

print("\nDisk Allocation Complete.")

--- Initializing Disk with 50 blocks ---

Allocated File: 'assignment.pdf'
Index Block: 39
Indexed Blocks pointing to Data Blocks: [35, 21, 25, 19]

Allocated File: 'notes.txt'
Index Block: 23
Indexed Blocks pointing to Data Blocks: [4, 38]

Disk Allocation Complete.
